# Fog density and camera-only object detection
## Research question
Does explicit fog-density information change detection accuracy and total inference cost for **YOLO11n, YOLO11m and DETR**?

**Current status:** experiment setup, not a completed study. No fog-conditioned model has been trained in this workspace. Run All safely displays the protocol and input readiness; training is opt-in below.

The user's current scope is authoritative: **Foggy Cityscapes + nuScenes**. The supplied CMPE 249 proposal supplies context and model names; its KITTI/STF alternatives and dates are not instructions to execute.


## Context & methods
This first pilot trains a small RGB residual adapter whose hidden features receive a scalar through feature-wise scale/shift (FiLM). Pretrained detector weights remain frozen. The adapter is identity-initialized.

| Arm | Adapter input | Research role |
|---|---|---|
| `rgb_control` | Constant zero | Same adapter, training images, seed, epochs and detector; no fog information |
| `predicted_density` | Beta estimated from RGB by a separately trained CNN | Deployable camera-only candidate; estimator cost included |
| `oracle_density` | Known synthetic beta | Oracle experiment; unavailable for unlabeled real images |

This is **input conditioning**, not a fourth image channel or internal detector-backbone modification. It does not establish the performance of jointly fine-tuned detectors. That is a subsequent experiment after this pilot. A zero-density control still learns RGB adaptation; it is not the untouched pretrained model.

### Key assumptions
- All detectors receive the same square RGB resize (default 640 px), batch one, FP32, seven shared classes. This deliberately replaces native aspect-ratio preprocessing and must be disclosed with results.
- Foggy Cityscapes beta is atmospheric attenuation in m^-1, not a confidence score. Fog variants of a base scene never cross splits. Optional original clear images have beta=0.
- Density training and detector training use disjoint sequences. Official Cityscapes train is split by sequence hash into density training (20%), detector training (60%) and tuning (20%); official val is held out as final test. Proportions are approximate by sequence.
- nuScenes is a separate transfer diagnostic: projected 3D boxes are not native 2D labels or official nuScenes 3D mAP. Native beta remains unknown. Object visibility annotations are not fog labels.
- Freeze hyperparameters before inspecting final test results. At least three seeds for the final study. A short run establishes execution only; improvement is not guaranteed.


## Setup
Use the project kernel. Install `requirements-fog-research.txt` if needed. First model loads may download weights. Dataset downloads require your existing dataset access.

The supplied density CNN can be trained from local beta labels. Its scores are predictions of synthetic attenuation, not calibrated physical measurements of real-world fog. A literature-backed external estimator can replace it only after verifying units, training provenance and held-out calibration.


In [8]:
from pathlib import Path
import os, sys, json
from datetime import datetime, timezone
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
assert (ROOT / 'notebooks/_support/fog_research.py').is_file(), 'Launch from the project or notebooks directory'
os.environ['YOLO_CONFIG_DIR'] = str(ROOT / '.ultralytics')
sys.path.insert(0, str(ROOT / 'notebooks/_support'))
import torch
from fog_research import (MODEL_SPECS, ARMS, CLASSES, read_manifest, select_groups,
                          run_experiment, plot_comparison)
from fog_data import cityscapes_manifest, nuscenes_manifest
plt.rcParams.update({'font.size': 11, 'figure.dpi': 110})

CITY_ROOT = ROOT / 'data/fog/cityscapes'
NUSCENES_ROOT = ROOT / 'data/depth/datasets/nuscenes'
CITY_MANIFEST = ROOT / 'data/fog/cityscapes.json'
NUSCENES_MANIFEST = ROOT / 'data/fog/nuscenes_all.json'
PREPARE_CITY = False
PREPARE_NUSCENES = False
RUN_EXPERIMENT = False
DEVICE = 'cuda:0' if torch.cuda.is_available() else 'cpu'
IMAGE_SIZE = 640
SEEDS = (42, 43, 44)
DENSITY_EPOCHS = 10
ADAPTER_EPOCHS = 5
MAX_GROUPS_PER_SPLIT = None  # e.g. 2 for a pilot; None preserves full available coverage
NUSCENES_MAX_FRAMES = None   # all locally available train/val keyframes and six cameras; use 100 only for a smoke test
CHECKPOINTS = {name: (ROOT / ckpt if name.startswith('YOLO') else ckpt)
               for name, ckpt in MODEL_SPECS.items()}
print('Device:', DEVICE, '| training enabled:', RUN_EXPERIMENT)
display(pd.DataFrame([{'model': m, 'arm': a, 'status': 'not run'} for m in CHECKPOINTS for a in ARMS]))

Device: cuda:0 | training enabled: False


,model,arm,status
0,YOLO11n,rgb_control,not run
1,YOLO11n,predicted_density,not run
2,YOLO11n,oracle_density,not run
3,YOLO11m,rgb_control,not run
4,YOLO11m,predicted_density,not run
5,YOLO11m,oracle_density,not run
6,DETR,rgb_control,not run
7,DETR,predicted_density,not run
8,DETR,oracle_density,not run


## Data
### 1. Prepare local datasets
Expected Cityscapes directories: `leftImg8bit_foggy/{train,val}/{city}`, `gtFine/{train,val}/{city}` and optionally `leftImg8bit` for clear images. Instance masks supply box rectangles; rider merges into person. Crowd regions are retained for COCO evaluation; crowd-bearing images are excluded from adapter training to avoid incorrect background supervision.

Preparation refuses to overwrite a manifest. Change the output name if rebuilding. The nuScenes adapter uses every locally available official train/validation keyframe across all six cameras, assigns whole scenes to detector-train or held-out test, keeps visibility levels 3/4, and records missing sensor files. It does not synthesize fog; native nuScenes has no beta labels, so density-aware nuScenes training requires a paired synthetic-fog generation pass using validated depth.


In [9]:
if PREPARE_CITY:
    cityscapes_manifest(CITY_ROOT, CITY_MANIFEST)
if PREPARE_NUSCENES:
    nuscenes_manifest(NUSCENES_ROOT, NUSCENES_MANIFEST, max_frames=NUSCENES_MAX_FRAMES)

rows, readiness = [], []
for name, path in [('Foggy Cityscapes', CITY_MANIFEST), ('nuScenes', NUSCENES_MANIFEST)]:
    if path.is_file():
        loaded = read_manifest(path)
        rows.extend(loaded)
        readiness.append({'dataset': name, 'status': 'validated', 'images': len(loaded)})
    else:
        readiness.append({'dataset': name, 'status': 'manifest missing; prepare data above', 'images': None})
display(pd.DataFrame(readiness))
if rows:
    selected = []
    for dataset, split in sorted({(r['dataset'], r['split']) for r in rows}):
        selected.extend(select_groups([r for r in rows if (r['dataset'], r['split']) == (dataset, split)],
                                      MAX_GROUPS_PER_SPLIT))
    rows = selected
    display(pd.DataFrame(rows).groupby(['dataset', 'split'], dropna=False).agg(
        images=('id', 'count'), sequences=('group', 'nunique'), known_beta=('beta', 'count')))
else:
    print('No prepared input yet. Remaining cells show the research workflow without invented results.')

,dataset,status,images
0,Foggy Cityscapes,manifest missing; prepare data above,None
1,nuScenes,manifest missing; prepare data above,None


No prepared input yet. Remaining cells show the research workflow without invented results.


### 2. Check leakage and density coverage
Input validation rejects duplicate image identities/paths, invalid boxes, nonfinite beta, missing images, unknown classes and sequences crossing splits. All fog variants should be present in every evaluated density bucket; inspect the table before training. Different bucket class populations can change mean AP even without model improvement.


In [10]:
if rows:
    coverage = pd.DataFrame(rows)
    display(coverage.groupby(['dataset', 'split', 'beta'], dropna=False).size().rename('images').reset_index())
    city_rows = [r for r in rows if r['dataset'] == 'foggy_cityscapes']
    variants = {}
    for row in city_rows:
        variants.setdefault(row['id'].rsplit('/', 1)[0], set()).add(row['beta'])
    missing_variants = [base for base, betas in variants.items() if not {.005, .01, .02}.issubset(betas)]
    assert not missing_variants, f'Incomplete fog variants: {missing_variants[:5]}'
    print('Shared classes:', ', '.join(CLASSES))

## Experiments
### 3. Train the density regressor and matched adapters, then evaluate
The loop runs sequentially to avoid GPU contention. It trains density only on `density_train`, freezes that model, and trains each adapter on identical `detector_train` rows. Every arm starts from the same adapter initialization per seed. Losses are saved each epoch in history tables, with final weights saved after training.

End-to-end latency starts before image decode and includes resize, transfer, fog estimation, adapter, detector and postprocessing. Three warm-ups are excluded; repeats use CUDA synchronization. Disk cache state and sequential model order can affect timing, so rerun with reversed model order before strong speed claims. Serial FPS is 1000/mean milliseconds, not video-pipeline throughput.

Outputs use a unique run directory and include settings, source hashes, image hashes, loaded detector hashes, exact rows, density errors, training curves, per-image boxes/timings, adapters, status and summaries. An interrupted run preserves completed exports; restarting requires a fresh run directory. Failed arms never receive zero accuracy.


In [11]:
results = pd.DataFrame()
status = pd.DataFrame()
RUN_DIR = None
if RUN_EXPERIMENT:
    if not any(r['dataset'] == 'foggy_cityscapes' for r in rows):
        raise RuntimeError('Prepare Foggy Cityscapes before training. nuScenes alone has no beta labels.')
    RUN_DIR = ROOT / 'reports/fog' / datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S-%fZ')
    results, status = run_experiment(rows, RUN_DIR, CHECKPOINTS, device=DEVICE, size=IMAGE_SIZE,
        seeds=SEEDS, density_epochs=DENSITY_EPOCHS, adapter_epochs=ADAPTER_EPOCHS)
    display(status)
    print('Saved:', RUN_DIR)
else:
    print('Training is off. Prepare manifests, review coverage, then set RUN_EXPERIMENT=True.')

Training is off. Prepare manifests, review coverage, then set RUN_EXPERIMENT=True.


## Results
### 4. Accuracy, density-estimation quality and latency
COCO bbox mAP@[.50:.95], AP50 and AR100 use the same seven categories and max 100 detections per image. Absent classes are excluded by COCO; a -1 metric means unsupported, not negative accuracy. Keep datasets separate because label definitions and class populations differ. Crowd treatment uses box regions, so this is a defined research benchmark, not an official Cityscapes leaderboard score.


In [12]:
if not results.empty:
    display(results.round(4))
    display(pd.read_csv(RUN_DIR / 'density-quality.csv').round(6))
else:
    print('No measured detector or density results are available yet.')

No measured detector or density results are available yet.


### 5. Compare accuracy versus speed and across fog levels
Each accuracy/latency point is one seed. Density curves average seeds and do not show confidence intervals. Do not infer significance from these plots. Known-density and estimated-density arms remain explicitly labeled.


In [13]:
plot_comparison(results)

No measured results. Prepare data and run experiments; no placeholder scores are plotted.


### 6. Paired changes from the no-density control
Positive delta mAP means improved accuracy; positive delta latency means slower inference. Only completed, matched model/dataset/seed/density pairs enter this table. Missing comparisons stay missing.


In [14]:
if not results.empty:
    keys = ['model', 'dataset', 'seed', 'density']
    control = results[results.arm == 'rgb_control'][keys + ['mAP50_95', 'mean_ms']]
    paired = results[results.arm != 'rgb_control'].merge(control, on=keys, how='inner',
        suffixes=('', '_control'), validate='many_to_one')
    paired['delta_mAP'] = paired.mAP50_95 - paired.mAP50_95_control
    paired['delta_latency_ms'] = paired.mean_ms - paired.mean_ms_control
    display(paired[keys + ['arm', 'delta_mAP', 'delta_latency_ms']].round(4))
    paired.to_csv(RUN_DIR / 'paired-deltas.csv', index=False)
else:
    print('Paired differences require completed control and density runs on the same images.')

Paired differences require completed control and density runs on the same images.


## Takeaways and next experiments
**No conclusion about improvement is available until training and evaluation complete.** Report negative results as well as positive ones.

1. Examine density MAE/RMSE, example detections and training losses before interpreting detector differences. Predicted beta on nuScenes has no ground-truth calibration in this setup.
2. Compare predicted-density against the matched control for practical benefit; compare oracle against predicted-density to assess estimator limitations.
3. Add constant-mean and shuffled-density controls, an untouched pretrained reference and jointly fine-tuned detector arms. Internal backbone FiLM or query conditioning is a distinct follow-up, not implemented here.
4. Add paired scene-bootstrap confidence intervals to AP deltas and repeat seeds/model execution order before claiming robust gains. Full fine-tuning, external pretrained fog-estimator validation and synthetic nuScenes fog generation remain future work.

### Sources
- [Foggy Cityscapes authors: beta values, imagery and annotations](https://people.ee.ethz.ch/~csakarid/SFSU_synthetic/)
- [Ultralytics YOLO11 model family](https://docs.ultralytics.com/models/yolo11/)
- [Hugging Face DETR](https://huggingface.co/docs/transformers/model_doc/detr)
- [nuScenes 2D projection reference](https://github.com/nutonomy/nuscenes-devkit/blob/master/python-sdk/nuscenes/scripts/export_2d_annotations_as_json.py)
- [Cityscapes labels](https://github.com/mcordts/cityscapesScripts/blob/master/cityscapesscripts/helpers/labels.py)
- Local context: `CMPE 249 Project Ideas.pdf`, supplied by the user. Current request overrides its older dataset alternatives.
